# Drift Characterization — TOPAs Phase Shifters

Characterize the thermal drift observed when applying electrical power to the TOPAs.
- Flux drops over ~2 min after powering ON
- Slow recovery over ~30 min after powering OFF

## §0 — Setup & Helpers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from datetime import datetime
from phobos import *

In [ ]:
# Output directory
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = os.path.join('generated', 'drift_characterization', TIMESTAMP)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

In [ ]:
def monitor_flux(duration_s, interval_s=1.0, record_power=False, n_avg=5):
    """
    Monitor output fluxes over time.
    
    Returns
    -------
    timestamps : ndarray (N,)
    fluxes : ndarray (N, 4)
    powers : ndarray (N, 4) or None
    """
    t0 = time.time()
    timestamps, fluxes, powers = [], [], []
    
    while (time.time() - t0) < duration_s:
        # Average n_avg frames
        frames = [Cred3().get_outputs() for _ in range(n_avg)]
        flux = np.mean(frames, axis=0)
        
        timestamps.append(time.time() - t0)
        fluxes.append(flux)
        
        if record_power:
            powers.append(Arch6().get_powers())
        
        # Wait for next sample
        elapsed = time.time() - t0
        next_t = len(timestamps) * interval_s
        if next_t > elapsed:
            time.sleep(next_t - elapsed)
    
    return (
        np.array(timestamps),
        np.array(fluxes),
        np.array(powers) if record_power else None
    )

In [ ]:
def plot_flux_curves(timestamps, fluxes, title, powers=None, save_as=None):
    """Plot flux (and optional power) curves."""
    colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']
    
    fig, ax1 = plt.subplots(figsize=(12, 5))
    for i in range(fluxes.shape[1]):
        ax1.plot(timestamps, fluxes[:, i], color=colors[i], label=f'Output {i}')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Flux (ADU)')
    ax1.set_title(title)
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    if powers is not None:
        ax2 = ax1.twinx()
        for i in range(powers.shape[1]):
            ax2.plot(timestamps, powers[:, i], '--', color=colors[i], alpha=0.5, label=f'TOPA {i} power')
        ax2.set_ylabel('Electrical Power (W)')
        ax2.legend(loc='upper right')
    
    plt.tight_layout()
    if save_as:
        fig.savefig(os.path.join(SAVE_DIR, save_as), dpi=150)
    plt.show()
    return fig


def save_results(name, **data):
    """Save results to .npz file."""
    path = os.path.join(SAVE_DIR, f'{name}.npz')
    np.savez(path, **data)
    print(f'Saved: {path}')
    return path

## §1 — Baseline (no TOPAs)

In [ ]:
# Injection calibration
DM().calibrate_injection()
DM().max()

In [ ]:
# Make sure all TOPAs are OFF
Arch6().turn_off()

# Record baseline flux for 5 min
print('Recording baseline (5 min, no TOPAs)...')
t_base, f_base, _ = monitor_flux(300, interval_s=1)

plot_flux_curves(t_base, f_base, 'Baseline — No TOPAs', save_as='01_baseline.png')
save_results('01_baseline', timestamps=t_base, fluxes=f_base)

## §2 — Flux Drop at Different Powers

Try different powers and compare the flux drop curve.

In [ ]:
TEST_POWERS = [0.1, 0.3, 0.5, 0.7, 1.0]  # Watts
STABILIZE_S = 300   # 5 min stabilization
DROP_S = 180        # 3 min drop monitoring
RECOVERY_S = 1800   # 30 min recovery monitoring

drop_results = {}

for P in TEST_POWERS:
    print(f'\n{"="*60}')
    print(f'Testing P = {P} W')
    print(f'{"="*60}')
    
    # 1. Turn off & stabilize
    Arch6().turn_off()
    print(f'  Stabilizing ({STABILIZE_S}s)...')
    time.sleep(STABILIZE_S)
    
    # 2. Measure baseline (30s)
    print('  Measuring baseline...')
    t_bl, f_bl, _ = monitor_flux(30, interval_s=1)
    baseline = np.mean(f_bl, axis=0)
    
    # 3. Power ON — monitor drop
    print(f'  Powering ON at {P} W — monitoring drop ({DROP_S}s)...')
    Arch6().set_powers([P] * 4)
    t_drop, f_drop, _ = monitor_flux(DROP_S, interval_s=1)
    
    # 4. Power OFF — monitor recovery
    print(f'  Powering OFF — monitoring recovery ({RECOVERY_S}s)...')
    Arch6().turn_off()
    t_rec, f_rec, _ = monitor_flux(RECOVERY_S, interval_s=2)
    
    drop_results[P] = {
        'baseline': baseline,
        't_drop': t_drop, 'f_drop': f_drop,
        't_recovery': t_rec, 'f_recovery': f_rec,
    }
    
    plot_flux_curves(t_drop, f_drop, f'Flux Drop — P={P} W', save_as=f'02_drop_P{P}.png')
    plot_flux_curves(t_rec, f_rec, f'Recovery — P={P} W', save_as=f'02_recovery_P{P}.png')
    
    save_results(f'02_power_{P}W',
        baseline=baseline,
        t_drop=t_drop, f_drop=f_drop,
        t_recovery=t_rec, f_recovery=f_rec)

In [ ]:
# Overlay normalized drop curves for all powers
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(TEST_POWERS)))

for idx, P in enumerate(TEST_POWERS):
    r = drop_results[P]
    # Normalize: divide by baseline total flux
    total_bl = np.sum(r['baseline'])
    
    # Drop phase
    total_drop = np.sum(r['f_drop'], axis=1) / total_bl
    axes[0].plot(r['t_drop'], total_drop, color=cmap[idx], label=f'{P} W')
    
    # Recovery phase
    total_rec = np.sum(r['f_recovery'], axis=1) / total_bl
    axes[1].plot(r['t_recovery'] / 60, total_rec, color=cmap[idx], label=f'{P} W')

axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Normalized Total Flux')
axes[0].set_title('Drop Phase (normalized)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Time (min)'); axes[1].set_ylabel('Normalized Total Flux')
axes[1].set_title('Recovery Phase (normalized)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '02_overlay.png'), dpi=150)
plt.show()

## §3 — Small Powers: Threshold Detection

Try small powers to see if the flux stops dropping at some point.

In [ ]:
SMALL_POWERS = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2]  # Watts

threshold_results = {}

for P in SMALL_POWERS:
    print(f'\nTesting small power P = {P} W')
    
    Arch6().turn_off()
    print(f'  Stabilizing ({STABILIZE_S}s)...')
    time.sleep(STABILIZE_S)
    
    # Baseline
    t_bl, f_bl, _ = monitor_flux(30, interval_s=1)
    baseline = np.mean(f_bl, axis=0)
    
    # Power ON
    print(f'  Powering ON — monitoring ({DROP_S}s)...')
    Arch6().set_powers([P] * 4)
    t_drop, f_drop, _ = monitor_flux(DROP_S, interval_s=1)
    Arch6().turn_off()
    
    threshold_results[P] = {'baseline': baseline, 't': t_drop, 'f': f_drop}
    save_results(f'03_small_P{P}W', baseline=baseline, t=t_drop, f=f_drop)

In [ ]:
# Overlay + compute relative drop amplitude
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
cmap = plt.cm.plasma(np.linspace(0, 1, len(SMALL_POWERS)))

drop_amplitudes = []

for idx, P in enumerate(SMALL_POWERS):
    r = threshold_results[P]
    total_bl = np.sum(r['baseline'])
    total = np.sum(r['f'], axis=1) / total_bl
    ax1.plot(r['t'], total, color=cmap[idx], label=f'{P} W')
    
    # Drop amplitude: (baseline - min) / baseline
    drop_amp = (1.0 - np.min(total))
    drop_amplitudes.append(drop_amp)

ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Normalized Total Flux')
ax1.set_title('Drop at Small Powers'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(SMALL_POWERS, drop_amplitudes, 'o-', color='#e63946')
ax2.set_xlabel('Power (W)'); ax2.set_ylabel('Relative Drop Amplitude')
ax2.set_title('Drop Amplitude vs Power'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '03_threshold.png'), dpi=150)
plt.show()

## §4 — Record Electrical Power During Drop

Record also the injected electrical power during the flux drop.

In [ ]:
P_TEST = 0.5  # W

Arch6().turn_off()
print(f'Stabilizing ({STABILIZE_S}s)...')
time.sleep(STABILIZE_S)

# Baseline
t_bl, f_bl, _ = monitor_flux(30, interval_s=1)
baseline_ep = np.mean(f_bl, axis=0)

# Power ON with power recording
print(f'Powering ON at {P_TEST} W — recording electrical power...')
Arch6().set_powers([P_TEST] * 4)
t_ep, f_ep, p_ep = monitor_flux(DROP_S, interval_s=1, record_power=True)
Arch6().turn_off()

plot_flux_curves(t_ep, f_ep, f'Flux + Electrical Power — P={P_TEST} W',
                 powers=p_ep, save_as='04_elec_power.png')
save_results('04_elec_power', baseline=baseline_ep, t=t_ep, f=f_ep, powers=p_ep)

## §5 — Injection Re-calibration During Drop

Perform injection calibration several times during the drop to see if the Gaussian shifts.

In [ ]:
Arch6().turn_off()
print(f'Stabilizing ({STABILIZE_S}s)...')
time.sleep(STABILIZE_S)

# Initial calibration
DM().calibrate_injection(grid_n=10)
DM().max()

# Record initial tip/tilt positions
initial_ptt = {seg_id: DM()[seg_id].get_ptt() for seg_id in Config().get('dm.injection_segments')}
print(f'Initial positions: {initial_ptt}')

# Power ON
P_CAL = 0.5
print(f'Powering ON at {P_CAL} W...')
Arch6().set_powers([P_CAL] * 4)

# Re-calibrate at regular intervals
RECAL_TIMES = [0, 30, 60, 90, 120]  # seconds after power ON
cal_results = []
t0 = time.time()

for target_t in RECAL_TIMES:
    # Wait until target time
    wait = target_t - (time.time() - t0)
    if wait > 0:
        time.sleep(wait)
    
    actual_t = time.time() - t0
    print(f'\n  Re-calibrating at t={actual_t:.0f}s...')
    
    # Quick calibration (small grid)
    DM().calibrate_injection(grid_n=10)
    DM().max()
    
    # Record new positions
    ptt = {seg_id: DM()[seg_id].get_ptt() for seg_id in Config().get('dm.injection_segments')}
    flux_after = Cred3().get_outputs()
    
    cal_results.append({
        'time': actual_t,
        'ptt': ptt,
        'flux': flux_after
    })
    print(f'    Flux: {flux_after}')
    print(f'    Positions: {ptt}')

Arch6().turn_off()

In [ ]:
# Plot tip/tilt evolution
seg_ids = Config().get('dm.injection_segments')
times_cal = [r['time'] for r in cal_results]

fig, axes = plt.subplots(len(seg_ids), 2, figsize=(14, 3 * len(seg_ids)), sharex=True)
if len(seg_ids) == 1:
    axes = axes[np.newaxis, :]

for row, seg_id in enumerate(seg_ids):
    tips = [r['ptt'][seg_id][1] for r in cal_results]   # tip = index 1
    tilts = [r['ptt'][seg_id][2] for r in cal_results]  # tilt = index 2
    
    axes[row, 0].plot(times_cal, tips, 'o-', color='#e63946')
    axes[row, 0].set_ylabel(f'Seg {seg_id}\nTip (mrad)')
    axes[row, 0].grid(True, alpha=0.3)
    
    axes[row, 1].plot(times_cal, tilts, 'o-', color='#457b9d')
    axes[row, 1].set_ylabel(f'Tilt (mrad)')
    axes[row, 1].grid(True, alpha=0.3)

axes[0, 0].set_title('Tip Evolution')
axes[0, 1].set_title('Tilt Evolution')
axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')

plt.suptitle('Injection Position Shift During Drift', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '05_injection_shift.png'), dpi=150, bbox_inches='tight')
plt.show()

## §6 — Fractioned Scan (10×10 instead of 100)

Instead of 100 continuous points, take 10 blocks of 10 points to track drift during a scan.

In [ ]:
N_BLOCKS = 10
N_PER_BLOCK = 10
N_AVG = 5
PHASE_RANGE = np.linspace(0, 2 * np.pi, N_BLOCKS * N_PER_BLOCK)

# Stabilize
Arch6().turn_off()
DM().calibrate_injection()
DM().max()
time.sleep(60)

# Use shifter 0 (first TOPA), all inputs ON
DM().flat()  # All inputs active
shifter = Arch6().shifters[0]

block_data = []
block_times = []
t0 = time.time()

for b in range(N_BLOCKS):
    block_start = time.time() - t0
    phases_block = PHASE_RANGE[b * N_PER_BLOCK : (b + 1) * N_PER_BLOCK]
    block_fluxes = []
    
    for phi in phases_block:
        shifter.set_phase(phi)
        frames = [Cred3().get_outputs() for _ in range(N_AVG)]
        block_fluxes.append(np.mean(frames, axis=0))
    
    block_end = time.time() - t0
    block_data.append(np.array(block_fluxes))
    block_times.append((block_start, block_end))
    print(f'  Block {b+1}/{N_BLOCKS}: t=[{block_start:.1f}, {block_end:.1f}]s')

Arch6().turn_off()

In [ ]:
# Compare fractioned scan vs continuous
all_fluxes = np.concatenate(block_data, axis=0)  # (100, 4)
colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Full scan
for i in range(4):
    ax1.plot(PHASE_RANGE, all_fluxes[:, i], color=colors[i], label=f'Output {i}')
ax1.set_xlabel('Phase (rad)'); ax1.set_ylabel('Flux')
ax1.set_title('Fractioned Scan (10×10)'); ax1.legend(); ax1.grid(True, alpha=0.3)

# Mean flux per block (drift indicator)
mean_per_block = [np.mean(np.sum(bd, axis=1)) for bd in block_data]
block_centers = [0.5 * (t[0] + t[1]) for t in block_times]
ax2.plot(block_centers, mean_per_block, 'o-', color='#264653')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Mean Total Flux')
ax2.set_title('Drift During Scan'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '06_fractioned_scan.png'), dpi=150)
plt.show()

save_results('06_fractioned_scan',
    phases=PHASE_RANGE, fluxes=all_fluxes,
    block_times=np.array(block_times), mean_per_block=np.array(mean_per_block))

## §7 — Minimize TOPA Power with DM Piston Compensation

Use DM pistons for phase compensation instead of TOPAs, to minimize thermal load.
Compare TOPA-only vs DM-compensated null calibration.

In [ ]:
# --- TOPA-only calibration ---
Arch6().turn_off()
DM().calibrate_injection()
DM().max()
time.sleep(60)

print('=== TOPA-only null calibration ===')
res_topa = Arch6().null_calibration_gen(verbose=True, plot=True)

# Monitor flux after TOPA calibration
print('Monitoring post-TOPA-calibration flux (3 min)...')
t_topa, f_topa, _ = monitor_flux(DROP_S, interval_s=1)
topa_powers = Arch6().get_powers()
print(f'TOPA powers after calibration: {topa_powers}')

In [ ]:
# --- DM-compensated calibration ---
Arch6().turn_off()
DM().calibrate_injection()
DM().max()
time.sleep(60)

print('=== DM-compensated null calibration ===')
res_dm = Arch6().null_calibration_gen(use_dm=True, lam=1550.0, verbose=True, plot=True)

# Monitor flux after DM calibration (TOPAs should be off or minimal)
print('Monitoring post-DM-calibration flux (3 min)...')
t_dm, f_dm, _ = monitor_flux(DROP_S, interval_s=1)
dm_topa_powers = Arch6().get_powers()
print(f'TOPA powers after DM calibration: {dm_topa_powers}')

In [ ]:
# Compare
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(t_topa, np.sum(f_topa, axis=1), label='TOPA-only', color='#e63946')
ax1.plot(t_dm, np.sum(f_dm, axis=1), label='DM-compensated', color='#457b9d')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Total Flux')
ax1.set_title('Post-Calibration Flux Stability'); ax1.legend(); ax1.grid(True, alpha=0.3)

# Bar chart of TOPA power usage
x = np.arange(4)
w = 0.35
ax2.bar(x - w/2, topa_powers, w, label='TOPA-only', color='#e63946')
ax2.bar(x + w/2, dm_topa_powers, w, label='DM-compensated', color='#457b9d')
ax2.set_xlabel('TOPA index'); ax2.set_ylabel('Power (W)')
ax2.set_title('TOPA Power Usage'); ax2.legend(); ax2.set_xticks(x); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '07_dm_compensation.png'), dpi=150)
plt.show()

save_results('07_dm_compensation',
    t_topa=t_topa, f_topa=f_topa, topa_powers=np.array(topa_powers),
    t_dm=t_dm, f_dm=f_dm, dm_topa_powers=np.array(dm_topa_powers))

## §8 — Null Measurement with Atmospheric Turbulence

Introduce atmospheric turbulence for null measurements.

In [ ]:
# First: null calibration (DM-based to minimize drift)
Arch6().turn_off()
DM().calibrate_injection()
DM().max()
time.sleep(60)

res_null = Arch6().null_calibration_gen(use_dm=True, lam=1550.0, verbose=True, plot=True)
print(f'Null-depth achieved: {res_null["depth_evol"][-1]:.2e}')

In [ ]:
# Turbulence parameters
atmo_params = {
    'r0': 0.15,           # Fried parameter (m)
    'wind_speed': 10.0,   # m/s
    'duration': 5.0,      # seconds
    'time_step': 0.005,   # 5 ms
    'wavelength': 1.65e-6 # H-band
}

# DM segment indices for the 2-input fringe tracking
dm_segs = Config().get('dm.injection_segments')
dm_pair = (dm_segs[0], dm_segs[1])

print('Running ABCD fringe tracking with turbulence...')
tracking_result = Arch6().abcd_fringe_tracking(
    atmosphere_params=atmo_params,
    dm_segments_indices=dm_pair,
    input_indices=(0, 1)
)

print('Done.')

In [ ]:
# Also try the analytical prediction (no hardware) for comparison
print('Analytical prediction with turbulence...')
pred_result = Arch6().predict_abcd_fringe_tracking(
    atmosphere_params=atmo_params,
    input_indices=(0, 1)
)
print('Done.')

## Summary

All drift characterization data has been saved to the `generated/drift_characterization/` directory.

In [ ]:
print(f'\nAll results saved in: {SAVE_DIR}')
print('Files:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f'  {f:40s} {size:.1f} KB')